# Chapter 11: Replication and Consistency
*From: Database Internals*

## TL;DR

- **Replication buys availability**, but keeping N copies in sync is equivalent to consensus — so we build a **spectrum of consistency models** that relax the "one global clock" fantasy in exchange for latency, availability, and cost.
- **CAP** says: under a network partition you must give up either consistency (linearizability) or availability; **PACELEC** adds: even without a partition you still trade latency for consistency. **Harvest/yield** reframes the binary into tunable degrees of completeness.
- The single-object ladder, strongest to weakest, is **Strict -> Linearizable -> Sequential -> Causal -> PRAM/Session -> Eventual -> Strong Eventual (SEC)**. Only linearizability composes; the others require system-wide care.
- **Tunable consistency** picks N, W, R such that `R + W > N` to guarantee read-write overlap; **witness replicas** split the N replicas into copies + witnesses to reduce storage while preserving quorum safety.
- **CRDTs** sidestep coordination entirely: if the merge is commutative, associative, and idempotent, concurrent updates converge deterministically — you get strong eventual consistency *without* consensus.

## Why Consistency Models

Every operation has an **invocation** and a **completion** event. Two operations are *sequential* if one completes before the other starts; otherwise they are *concurrent*. A storage cell accessed by read/write is a **register**, and registers come in three flavors:

- **Safe** — concurrent read during a write may return any value in range (think flickering bits).
- **Regular** — a read returns either the last completed write or a value from a concurrent write; order exists but propagation is lagged.
- **Atomic** — linearizable: every write has a single instant before which readers see the old value and after which they see the new one.

A consistency model is a **contract**: it limits which of the many possible interleavings are permissible. The weaker the model, the more interleavings are legal and the cheaper the implementation — but the harder it is to reason about. The rest of this chapter walks the spectrum from strongest (strict) to weakest (SEC), and then shows the two practical levers real databases pull: **quorum tuning (N/W/R)** and **CRDT merges**.

## Quorum Math — N, W, R, and Failure Tolerance

The single most load-bearing inequality in tunable consistency is `R + W > N`. When it holds, any read quorum and any write quorum must share at least one replica, so the read sees the latest completed write. Below we sweep realistic configurations and compute failure tolerance `f = (N-1)/2` (integer division — the system can lose up to `f` nodes and still form a majority quorum).

In [ ]:
# Quorum calculator: does R + W > N guarantee strong reads?
# Also computes majority failure tolerance f = (N-1) // 2.

configs = [
    # (N, W, R, label)
    (3, 2, 2, "classic majority (Dynamo default)"),
    (5, 3, 3, "5-node majority"),
    (5, 1, 5, "write-fast, read-all"),
    (5, 5, 1, "write-all, read-fast"),
    (3, 3, 1, "read-anywhere, write-all"),
    (3, 1, 1, "best-effort AP (no overlap)"),
    (7, 4, 4, "7-node majority"),
]

header = f"{'Config':<34} {'N':>2} {'W':>2} {'R':>2} {'R+W>N':>6} {'Strong read':>12} {'f (maj tol)':>11}"
print(header)
print("-" * len(header))
for N, W, R, label in configs:
    overlap = R + W > N
    strong = "yes" if overlap else "no"
    f = (N - 1) // 2
    # Write quorum needs at least W alive nodes; read quorum needs R alive nodes.
    # Strongest majority requires both W > N/2 and R > N/2.
    majority_both = (W > N // 2) and (R > N // 2)
    tol = f if majority_both else max(0, min(N - W, N - R))
    marker = "*" if majority_both else " "
    print(f"{label:<34} {N:>2} {W:>2} {R:>2} {strong:>6} {strong:>12} {tol:>10}{marker}")

print()
print("* = both W and R are strict majorities; system tolerates f = (N-1)//2 simultaneous failures")
print("  while still accepting reads and writes. Other rows trade availability on one side for the other.")

# Quick sanity: what does R+W>N cost you as N grows?
print()
print("Coordinator message cost for a quorum write (min nodes to wait on):")
for N in (3, 5, 7, 9):
    W = N // 2 + 1
    print(f"  N={N}: W={W} acks required, async to remaining {N - W}")

### A G-Counter CRDT converges regardless of merge order

A **grow-only counter** (G-Counter) is the "hello world" of CRDTs. Each of the N nodes owns one slot in a length-N vector and may only increment *its own* slot. The merge function is **pointwise max**, which is commutative, associative, and idempotent — so any gossip order produces the same final vector, and the counter value is simply the sum of slots. Below we run three nodes, apply a bunch of increments, gossip updates in *different* orders on each node, and confirm all three converge to the same total.

In [ ]:
# G-Counter CRDT: three nodes, pointwise-max merge, arbitrary gossip order.

from itertools import permutations

NODES = 3

def new_counter():
    return [0] * NODES

def inc(state, node_id, n=1):
    # Only the owning node may bump its own slot.
    new = list(state)
    new[node_id] += n
    return new

def merge(a, b):
    return [max(x, y) for x, y in zip(a, b)]

def value(state):
    return sum(state)

# Simulate local activity on three nodes.
n0 = inc(inc(new_counter(), 0, 3), 0, 2)   # node 0 bumps itself by 5
n1 = inc(new_counter(), 1, 7)              # node 1 bumps itself by 7
n2 = inc(inc(new_counter(), 2, 1), 2, 4)   # node 2 bumps itself by 5

local_states = [n0, n1, n2]
print("Local states after independent increments:")
for i, s in enumerate(local_states):
    print(f"  node {i}: {s}  (value = {value(s)})")

print()
print("Convergence check — merge the three states in every possible order:")
totals = set()
for order in permutations(range(NODES)):
    acc = new_counter()
    for idx in order:
        acc = merge(acc, local_states[idx])
    totals.add(tuple(acc))
    print(f"  merge order {order}: state={acc}, value={value(acc)}")

assert len(totals) == 1, "CRDT did not converge!"
print()
print(f"All orderings converged to the same state: {totals.pop()}  -> SEC holds.")

## The Consistency Spectrum

```mermaid
graph LR
  Strict["Strict<br/>(theoretical)"] --> Lin["Linearizable<br/>real-time + total order"]
  Lin --> Seq["Sequential<br/>per-process order + agreed total order"]
  Seq --> Causal["Causal<br/>happens-before preserved"]
  Causal --> PRAM["PRAM / Session<br/>FIFO per client"]
  PRAM --> Even["Eventual<br/>converges someday"]
  Even --> SEC["Strong Eventual<br/>CRDTs: deterministic convergence"]
```

Strength decreases left-to-right; synchronization cost and latency decrease too. Only **linearizability** composes — a system built from linearizable objects is itself linearizable. Every weaker model requires end-to-end reasoning.

## The Linearization Point — Real-Time Bounds

```mermaid
sequenceDiagram
  participant Client
  participant Register as Register x
  participant Reader
  Note over Register: initial x = 0
  Client->>Register: invoke write(x, 1) at t1
  Note over Register: linearization point<br/>(some t in [t1, t2])<br/>x becomes 1 atomically
  Register-->>Client: ack at t2
  Reader->>Register: read(x) at t >= t2
  Register-->>Reader: 1
  Note over Reader: any read after t2 must see 1<br/>or a later value
```

The write appears to take effect at a single instant between `t1` (invocation) and `t2` (completion). Past that point, no reader may observe the pre-write value — that is what makes the system "look like a single machine". In practice the linearization point is enforced by consensus, locks, or CAS on a designated authority replica.

## Quorum Overlap: N=3, W=2, R=2

```mermaid
graph TB
  subgraph "Write quorum W=2 (replicas acked)"
    W1[Replica A - wrote v2]
    W2[Replica B - wrote v2]
    W3[Replica C - still v1]
  end
  subgraph "Read quorum R=2 (any two)"
    R1[Replica A]
    R2[Replica C]
  end
  W1 -.same node.- R1
  W3 -.same node.- R2
  Note["R + W = 4 > 3 = N<br/>=> read quorum must contain<br/>at least one writer<br/>=> reader sees v2"]
  R1 --> Note
```

The pigeonhole: with three replicas, any two chosen for a write and any two chosen for a read must share at least one node. The shared node is the witness of the most recent value.

## CRDT Merge — Three Nodes Converge

```mermaid
graph LR
  subgraph "Local state after independent increments"
    A["Node 1: [5, 0, 0]"]
    B["Node 2: [0, 7, 0]"]
    C["Node 3: [0, 0, 5]"]
  end
  A -->|gossip| AB["merge: [5, 7, 0]"]
  B -->|gossip| AB
  AB --> ABC["merge with C:<br/>[5, 7, 5]<br/>sum = 17"]
  C --> ABC
  A -->|gossip| AC["merge: [5, 0, 5]"]
  C -->|gossip| AC
  AC --> ABC2["merge with B:<br/>[5, 7, 5]<br/>sum = 17"]
  B --> ABC2
  ABC -.same result.- ABC2
```

Different gossip paths, same final state. That is the whole CRDT trick: make the merge operator a mathematical lattice join, and ordering stops mattering.

## Deep Dive 1: CAP, PACELEC, and Harvest/Yield

CAP (Brewer) says that under a network partition a system must choose between **Consistency (linearizability)** and **Availability (every non-failed node responds)** — Partition tolerance is not a knob, it's a fact of life. **PACELEC** (Abadi) extends this: in the *absence* of a partition (E, "Else"), the remaining trade is between **Latency and Consistency**. CAP's "consistency" is narrower than ACID's (atomic/linearizable, not invariant-preserving), and CAP's "availability" is narrower than operational SLAs (any non-failed node, any request, eventually). **Harvest/yield** (Fox, Brewer) reframes the binary: *harvest* is how complete a response is (e.g. 99 of 100 rows) and *yield* is how many requests succeeded; real systems tune both rather than flipping a C/A switch.

## Deep Dive 2: Linearizability

The strongest single-object model: every operation appears to take effect at a single **linearization point** inside its real-time bound `[t1, t2]`. Once a value is visible, no reader may go back to an older one — monotonicity is forced. Linearizability is **local/composable**: a system whose objects are individually linearizable is itself linearizable, which is why CPU atomics and `compare-and-swap` are so useful. The costs are real: cache coherence traffic inside a CPU, consensus rounds across a cluster. **RIFL** (Reusable Infrastructure for Linearizability) extends linearizable semantics to RPCs by giving each client a lease + monotonic sequence number and storing a completion object alongside the mutation, so retries return the cached result instead of re-applying.

```mermaid
sequenceDiagram
  participant W1 as Writer W1
  participant W2 as Writer W2
  participant R as Reader
  participant x
  Note over x: initial x = 0
  W1->>x: write(x, 1) during [t1..t2]
  W2->>x: write(x, 2) during [t1'..t2'] overlaps W1
  R->>x: read(x) returns 1 or 2 (both legal under overlap)
  R->>x: read(x) returns 2
  Note over R: after reading 2, must never read 1 again (monotonic)
```

## Deep Dive 3: Sequential Consistency

Sequential consistency **drops the real-time bound** but keeps two invariants: (1) each process's operations are seen in its own program order, and (2) **all** processes agree on one total order of all operations. Two readers can be arbitrarily stale relative to wall-clock time, yet they cannot disagree with each other. Unlike linearizability, sequential consistency is **not composable** — two sequentially consistent objects combined can produce non-sequential histories. Lamport introduced it for multiprocessor memory: think of it as "there exists *some* legal interleaving," even if that interleaving is later than wall clock.

```mermaid
sequenceDiagram
  participant P1
  participant P2
  participant P3
  participant P4
  participant x
  P1->>x: write(x, 1) (wall clock first)
  P2->>x: write(x, 2) (wall clock second)
  Note over P3,P4: Both readers agree on order 2 then 1.
  P3->>x: read returns 2
  P3->>x: read returns 1
  P4->>x: read returns 2
  P4->>x: read returns 1
```

## Deep Dive 4: Causal Consistency and Vector Clocks

Causal consistency demands only that **causally related** operations (one *happened-before* another) be observed in the same order by every process — concurrent writes may be seen in any order. Implementations attach a logical timestamp (often a **vector clock**) to each write, encoding "I depend on these earlier events." A replica receiving `write(x, t1, 2)` buffers it until it has already applied everything `t1` depended on. **COPS** tracks dependencies by key-version, **Eiger** by operation order, both deliver out-of-order-safe reads. Vector clocks (Dynamo, Riak) can *detect* conflicts but not resolve them — applications must merge divergent histories (or fall back to last-write-wins, as Cassandra does).

```mermaid
sequenceDiagram
  participant P1
  participant P2
  participant P3
  participant x
  P1->>x: write(x, 0, 1) yields t1
  P2->>x: write(x, t1, 2) yields t2 (depends on t1)
  Note over P3: receives write t2 first
  P3->>P3: buffer t2 (missing t1)
  Note over P3: later receives t1
  P3->>P3: apply t1, then apply t2
```

## Deep Dive 5: Session Models (Client-Centric Consistency)

Session models describe how a **single client** sees the system across a session, regardless of which replica it lands on. Four canonical guarantees:

- **Read-own-writes** — a client that just wrote V cannot read an older value.
- **Monotonic reads** — if a client reads V, subsequent reads return V or newer.
- **Monotonic writes** — a client's writes propagate to others in the order it made them.
- **Writes-follow-reads** — a write issued after reading V is ordered after the write that produced V.

Combining monotonic-reads, monotonic-writes, and read-own-writes yields **PRAM/FIFO consistency**: per-process FIFO ordering of writes, with no cross-process agreement. Session models are often what applications actually want — they are cheap to implement (sticky sessions, version tokens) and map cleanly to user-facing expectations.

## Deep Dive 6: Tunable Consistency and Witness Replicas

Quorum systems expose three knobs: replication factor **N**, write quorum **W**, read quorum **R**. Setting `R + W > N` forces read-write overlap; setting both `W` and `R` to `N/2 + 1` (a majority) tolerates `f = (N-1)/2` failures. Asymmetric settings trade availability: `W=1, R=N` is write-fast / read-paranoid; `W=N, R=1` is the inverse. **Witness replicas** cut storage cost: of N replicas, only `n` hold full copies and `m = N - n` are "witnesses" that normally store just presence metadata but can be **upgraded** to hold a value when a copy replica is unavailable. Availability matches `n + m` full replicas, storage cost is only `n` — used in Spanner and Cassandra.

```mermaid
graph LR
  Client --> Coord[Coordinator]
  Coord -->|write x=v| C1[Copy 1c]
  Coord -->|write x=v| C2[Copy 2c down]
  Coord -->|write x=v| W3[Witness 3w<br/>upgraded: holds v]
  C1 -.ack.-> Coord
  W3 -.ack.-> Coord
  Coord -->|W=2 satisfied| Client
```

## Deep Dive 7: CRDTs and Strong Eventual Consistency

A **Conflict-Free Replicated Data Type** is a data structure whose merge operation is commutative, associative, and idempotent — a **join-semilattice**. Any two replicas that have seen the same set of updates converge to the same state, regardless of delivery order (**Strong Eventual Consistency**, SEC). The canonical examples:

- **G-Counter** — per-node increment slots, merge = pointwise max.
- **PN-Counter** — two G-Counters (P for increments, N for decrements).
- **LWW-Register** — value with timestamp; merge keeps the larger timestamp.
- **Multi-Value Register** — retains all concurrent writes; app resolves.
- **G-Set / 2P-Set** — grow-only set; 2P-Set pairs an add-set with a remove-set.
- **JSON CRDT** — nested maps and lists with client-side merges (Kleppmann).

CRDTs shine when you can accept an eventually-consistent view but cannot afford consensus latency per write — Redis active-active, collaborative editors (Automerge, Yjs), and mobile-first apps with long offline windows.

## Trade-offs

| Model | Global Order | Real-time Bound | Composable | Cost | Typical Use |
|---|---|---|---|---|---|
| **Strict** | Yes (instantaneous) | Hard | Yes | Impossible (requires global clock) | Theoretical baseline |
| **Linearizable** | Yes | Yes (between t1 and t2) | Yes | Consensus / CAS per op | Locks, leader state, single-key strong reads |
| **Sequential** | Yes | No | **No** | Per-process FIFO + agreement | Early shared-memory CPUs, some caches |
| **Causal** | Partial (happens-before) | No | No | Vector clocks + dep tracking | COPS, Eiger, Dynamo-style stores |
| **PRAM / Session** | Per-client only | No | No | Sticky sessions / version tokens | User-facing features (feeds, carts) |
| **Eventual** | None | No | No | Async gossip + conflict resolution | Cassandra, DynamoDB (default), DNS |
| **Strong Eventual (CRDT)** | None (but deterministic merge) | No | Yes on the data type | Metadata overhead in the CRDT | Collaborative editing, counters, offline-first |

## Key Takeaways

1. **Replication is the source of all consistency pain.** If there were always exactly one copy, there would be no models to argue about — CAP, PACELEC, and the whole spectrum exist because we keep N copies for availability.
2. **Linearizability is the only composable model.** Systems built from linearizable parts are linearizable; every other model requires end-to-end reasoning, which is why stacking weaker layers under stronger ones creates silent data loss.
3. **`R + W > N` is the load-bearing invariant of tunable consistency.** Majority reads and majority writes (`W = R = N/2 + 1`) tolerate `f = (N-1)/2` failures and give you strong reads without consensus on every op.
4. **Session guarantees are usually what users actually want.** Read-your-writes + monotonic reads cover most "why does the UI flicker" bugs, and they are much cheaper than linearizability.
5. **Causal consistency decouples logical order from wall-clock order.** Vector clocks let you detect concurrent updates without a global clock — but detection is not resolution; applications still need a merge strategy.
6. **Witness replicas buy availability at lower storage cost.** If you have copy + witness replicas and always include at least one copy in a quorum, you match the availability of `n+m` full replicas while paying only for `n` copies of the data.
7. **CRDTs are the uncoordinated alternative to consensus.** When the data type's merge is a lattice join, concurrent writes converge deterministically — no leader, no locks, no rollback — at the cost of richer metadata and some domain restrictions (you cannot CRDT arbitrary invariants).

## See Also

- [[01-cap-pacelec-and-harvest-yield]]
- [[02-linearizability]]
- [[03-sequential-consistency]]
- [[04-causal-consistency-and-vector-clocks]]
- [[05-session-models]]
- [[06-tunable-consistency-and-witness-replicas]]
- [[07-crdts-and-strong-eventual-consistency]]